In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv) 
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

import re
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/google/gemma/transformers/2b-it/2/model.safetensors.index.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/gemma-2b-it.gguf
/kaggle/input/models/google/gemma/transformers/2b-it/2/config.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/model-00001-of-00002.safetensors
/kaggle/input/models/google/gemma/transformers/2b-it/2/model-00002-of-00002.safetensors
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer_config.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/special_tokens_map.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/.gitattributes
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer.model
/kaggle/input/models/google/gemma/transformers/2b-it/2/generation_config.json
/kaggle/input/competitions/african-folktales-slm-challenge/sample_submission.csv
/kaggle/input/competitions/african-folktales-slm-challenge/train_prompts.cs

In [2]:
DATA_DIR = "/kaggle/input/competitions/african-folktales-slm-challenge"

documents = pd.read_csv(
    os.path.join(DATA_DIR, "documents.csv")
)

train = pd.read_csv(
    os.path.join(DATA_DIR, "train_prompts.csv")
)

test = pd.read_csv(
    os.path.join(DATA_DIR, "test_prompts.csv")
)



In [3]:
DIR = "/kaggle/input/notebooks/davidattah/document-retrieval"
retrieved_df = pd.read_csv(
    os.path.join(DIR, "submission_retrieval.csv")
)

In [4]:


print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [5]:
def create_training_text(row):

    return f"""### Prompt:
{row['prompt']}

### Theme:
{row['theme']}

### Region:
{row['culture_region']}

### Story:
{row['reference_story']}"""



train["text"] = train.apply(
    create_training_text,
    axis=1
)

print(train["text"].iloc[0])

### Prompt:
Explain in story form why the baobab looks upside down.

### Theme:
origin_myth

### Region:
east_africa

### Story:
The first baobab boasted that its roots could drink any star. The soil spirit grew tired of pride and planted the tree head-down so its branches learned humility underground. When travelers rest beneath its wide trunk, they remember: greatness must bow to the place that feeds it.


In [6]:
document_texts = []

for _, row in documents.iterrows():
    
    text = f"""### Prompt:
{row['title']}

### Theme:
{row['theme']}

### Region:
{row['culture_region']}

### Story:
{row['text']}"""

    document_texts.append(text)

print(document_texts[0])

### Prompt:
The spider and the shared pot

### Theme:
trickster

### Region:
west_africa

### Story:
When famine visited the village, the spider trickster proposed a shared cooking pot. Each family would contribute one grain, yet he stirred with a hollow ladle and ate the thickened stew alone. The elders watched, replaced the pot with two smaller ones, and said: cunning fills one belly while trust feeds the town. From that day, feasts began with open ladles and counted spoons.


In [7]:
# dataset = Dataset.from_pandas(
#     train[["text"]]
# )

# print(dataset)


prompt_texts = train["text"].tolist()

all_texts = (
    prompt_texts +
    document_texts
)

dataset = Dataset.from_dict({
    "text": all_texts
})

print(dataset[50])

{'text': "### Prompt:\nHyena and the moon's reflection\n\n### Theme:\nanimal_fable\n\n### Region:\neast_africa\n\n### Story:\nHyena saw the moon in a still pond and leapt to seize it, soaking his muzzle in mud. Owl hooted that some lights are for watching, not eating. Hyena laughed, washed, and guarded the pond so calves could drink by starlight. Knowing what you cannot catch is also a kind of feast."}


In [8]:
MODEL_PATH = "/kaggle/input/models/google/gemma/transformers/2b-it/2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

if torch.cuda.is_available():
    model = model.cuda()

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)




In [10]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [11]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765


In [12]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/folktale-lora",

    num_train_epochs=8,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),

    logging_steps=5,

    save_strategy="epoch",

    report_to="none",

    remove_unused_columns=False
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
5,4.721104
10,2.882311
15,2.045170
20,1.280804
25,0.754189
30,0.329074
35,0.209871
40,0.138836
45,0.095280
50,0.089991


TrainOutput(global_step=64, training_loss=0.9960160520859063, metrics={'train_runtime': 52.4523, 'train_samples_per_second': 9.456, 'train_steps_per_second': 1.22, 'total_flos': 527194063257600.0, 'train_loss': 0.9960160520859063, 'epoch': 8.0})

In [15]:
OUTPUT_DIR = "/kaggle/working/folktale-lora"

model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

('/kaggle/working/folktale-lora/tokenizer_config.json',
 '/kaggle/working/folktale-lora/chat_template.jinja',
 '/kaggle/working/folktale-lora/tokenizer.json')

In [16]:
def generate_story(
    prompt,
    theme,
    region,
    max_new_tokens=120
):

    text = f"""### Prompt:
{prompt}

### Theme:
{theme}

### Region:
{region}

### Story:
"""

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    if torch.cuda.is_available():
        inputs = {
            k: v.cuda()
            for k, v in inputs.items()
        }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return generated

In [17]:
row = train.iloc[1]

story = generate_story(
    row["prompt"],
    row["theme"],
    row["culture_region"]
)

print(story)

### Prompt:
Tell a story showing generosity warms a communal pot.

### Theme:
moral_tale

### Region:
east_africa

### Story:
A traveler asked for one ladle of stew and was refused by a stingy host. The communal pot in the next compound welcomed him, and steam rose at once. Villagers say generosity warms the stones beneath a meal; greed cools even a full hearth.


In [18]:
def extract_story(text):

    if "### Story:" in text:

        text = text.split(
            "### Story:",
            1
        )[1]

    # Remove accidental sections
    for marker in [
        "### Prompt:",
        "### Theme:",
        "### Region:"
    ]:

        if marker in text:
            text = text.split(
                marker,
                1
            )[0]

    return text.strip()


story = extract_story(story)

print(story)

A traveler asked for one ladle of stew and was refused by a stingy host. The communal pot in the next compound welcomed him, and steam rose at once. Villagers say generosity warms the stones beneath a meal; greed cools even a full hearth.


In [19]:
def generate_candidates(
    prompt,
    theme,
    region,
    n=5
):

    candidates = []

    for _ in range(n):

        story = generate_story(
            prompt,
            theme,
            region
        )

        story = extract_story(story)

        candidates.append(story)

    return candidates



# row = test.iloc[1]

# candidates = generate_candidates(
#     row["prompt"],
#     row["theme"],
#     row["culture_region"],
#     n=10
# )

# for i, story in enumerate(candidates):

#     print("=" * 60)
#     print(f"CANDIDATE {i+1}")
#     print(story)

In [20]:
all_candidates = []

for i, row in test.iterrows():

    prompt_id = row["PromptId"]


    candidates = [
        
    ]

    # LoRA generations
    generated = generate_candidates(
        row["prompt"],
        row["theme"],
        row["culture_region"],
        n=1
    )

    for story in generated:

        candidates.append({
            "source": "lora",
            "story": story
        })

    all_candidates.append({
        "PromptId": prompt_id,
        "prompt": row["prompt"],
        "theme": row["theme"],
        "culture_region": row["culture_region"],
        "candidates": candidates
    })

In [21]:
!pip install -q python-Levenshtein
import Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.0 MB/s eta 0:00:00


In [22]:
def get_relevant_references(prompt_id):

    relevant = retrieved_df[retrieved_df["PromptId"] == prompt_id]

    if len(relevant) == 0:

        return []

    return relevant["Story"].tolist()

In [23]:
def candidate_score(
    story,
    prompt_id
):

    references = get_relevant_references(
        prompt_id
    )

    if not references:
        return 999999

    distances = [
        Levenshtein.distance(
            story,
            str(reference)
        )
        for reference in references
    ]

    return np.mean(distances)

In [24]:
final_rows = []

for item in all_candidates:

    scored = []

    for candidate in item["candidates"]:

        story = candidate["story"]

        score = candidate_score(
            story,
            item["PromptId"]
        )

        scored.append({
            "source": candidate["source"],
            "story": story,
            "score": score
        })

    scored.sort(
        key=lambda x: x["score"]
    )

    best = scored[0]

    final_rows.append({
        "PromptId": item["PromptId"],
        "Story": best["story"],
        "source": best["source"],
        "score": best["score"]
    })

In [25]:
final_df = pd.DataFrame(final_rows)

display(final_df)

,PromptId,Story,source,score
0,1001,A merchant dropped a cowrie shell in the marke...,lora,272.0
1,1002,An orphan boy braided fish lines with stranger...,lora,267.0
2,1003,Hare borrowed thunder from a hollow log until ...,lora,224.0
3,1004,Hyena saw the moon in the water and leapt to s...,lora,51.0
4,1005,Monkey promised to guard the hive while bees m...,lora,0.0
5,1006,A daughter carried night in a covered calabash...,lora,182.0
6,1007,Grandmother sang beside a sleeping river and i...,lora,194.0
7,1008,Two brothers inherited one path to the grazing...,lora,40.0
8,1009,Gratitude became a drum and the sky learned to...,lora,221.0
9,1010,Villagers argued whose son would lead the harv...,lora,112.0


In [26]:
for _, row in final_df.iterrows():

    print("=" * 80)
    print("PromptId:", row["PromptId"])
    print("Source:", row["source"])
    print("Score:", row["score"])
    print()
    print(row["Story"])
    print()

PromptId: 1001
Source: lora
Score: 272.0

A merchant dropped a cowrie shell in the market dust and was mocked by a stingy seller. At sunset the shell owner visited the market and replaced the coin with interest, not coercion. The merchant followed, filled with regret, and bought the shell back at sunrise to prove honesty pays. He later gifted it to a sick child who used it to pay for medicine that cured him. In the marketplace no longer serves empty words but counted coins.

PromptId: 1002
Source: lora
Score: 267.0

An orphan boy braided fish lines with strangers in a squall and was repaid in kind. When a storm broke, he braided for elders alone, and they cast a net that day's catch aside for a girl's offering. Small offerings became braided lines; big offerings knots. He became chief of the harbor until a storm proved too strong for one rope. He taught children braided lines are knots, and a shared net feeds both shore and city.

PromptId: 1003
Source: lora
Score: 224.0

Hare borrowed

In [27]:
submission = final_df[
    ["PromptId", "Story"]
].copy()

In [28]:
submission_path = (
    "/kaggle/working/submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(
    f"Submission saved to: {submission_path}"
)

Submission saved to: /kaggle/working/submission.csv
